# Tutorial 04 - Project: Monthly Sales Trend & Forecasting Engine

Generates 12-month synthetic sales records for four products, computes
growth indicators, and plots seasonal patterns.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/07_Monthly_Sales_Forecasting/` project folder (anchored on `utils.py`; the generated CSVs go to the project's git-ignored `data/` folder) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "07_Monthly_Sales_Forecasting"
ANCHOR_REL = "utils.py"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Generate the Monthly Dataset


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils import generate_random_sales

np.random.seed(42)
os.makedirs(PROJ_DIR / "data", exist_ok=True)

# Generate 12 monthly dates for 2025
months = pd.date_range(start="2025-01-01", end="2025-12-01", freq="MS")
num_m = len(months)

df_initial = pd.DataFrame({
    "Date": months,
    "Product_A": generate_random_sales(50, 100, num_m),
    "Product_B": generate_random_sales(30, 80, num_m),
    "Product_C": generate_random_sales(20, 60, num_m),
    "Product_D": generate_random_sales(10, 50, num_m),
})

df_initial.to_csv(PROJ_DIR / "data" / "initial.csv", index=False)
display(df_initial.head())


### 2. Metric Calculations & Quarter Mapping


In [ ]:
df_final = df_initial.copy()
df_final["Month"] = df_final["Date"].dt.strftime("%b")
pcols = ["Product_A", "Product_B", "Product_C", "Product_D"]

df_final["Total_Sales"] = df_final[pcols].sum(axis=1)
df_final["Average_Sales"] = df_final[pcols].mean(axis=1)
df_final["MoM_Growth"] = df_final["Total_Sales"].pct_change() * 100

def assign_quarter(dt):
    m = dt.month
    if m <= 3: return "Q1"
    elif m <= 6: return "Q2"
    elif m <= 9: return "Q3"
    else: return "Q4"

df_final["Quarter"] = df_final["Date"].apply(assign_quarter)
df_final["Max_Product"] = df_final[pcols].idxmax(axis=1)
df_final["Min_Product"] = df_final[pcols].idxmin(axis=1)

df_final.to_csv(PROJ_DIR / "data" / "final.csv", index=False)
display(df_final.head())


### 3. Quarterly Summary Table


In [ ]:
# Quarterly summary - averages (handout) AND totals (issue #16)
pivot_q = df_final.pivot_table(index="Quarter", values=pcols + ["Total_Sales"], aggfunc="mean").round(2)
pivot_q_total = df_final.pivot_table(index="Quarter", values=pcols + ["Total_Sales"], aggfunc="sum")

# output.csv now carries both: the mean of each column and the total volume per quarter
pivot_q["Total_Volume"] = pivot_q_total["Total_Sales"]
pivot_q.to_csv(PROJ_DIR / "data" / "output.csv")

print("Quarterly averages (mean of monthly values):")
display(pivot_q.drop(columns=["Total_Volume"]))
print("Quarterly totals (sum of monthly values):")
display(pivot_q_total)

### 4. 4-Panel Visualization Grid


In [ ]:
sns.set_theme(style="whitegrid")
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# 1. Product Trends
for col in pcols:
    axs[0, 0].plot(df_final["Month"], df_final[col], marker="o", label=col)
axs[0, 0].set_title("Product Monthly Trends", fontweight="bold")
axs[0, 0].legend()

# 2. Stacked Sales Volume
df_final.set_index("Month")[pcols].plot(kind="bar", stacked=True, ax=axs[0, 1], colormap="viridis")
axs[0, 1].set_title("Monthly Total Sales Volume", fontweight="bold")

# 3. Density Heatmap
sns.heatmap(df_final.set_index("Month")[pcols], annot=True, fmt="d", cmap="YlGnBu", ax=axs[1, 0])
axs[1, 0].set_title("Monthly Sales Density", fontweight="bold")

# 4. Variance Boxplot (single colour -> no deprecation warning on seaborn >= 0.13)
sns.boxplot(data=df_final[pcols], color="lightgreen", ax=axs[1, 1])
axs[1, 1].set_title("Product Sales Distributions", fontweight="bold")

plt.tight_layout()
plt.show()


### 4. Key Insights (Best Month, Best Product, Best Quarter)

Printed explicitly, as the handout asks. On the seeded dataset (seed 42): the best month is
Feb with 226 units, the best product over the year is
Product_A (870 units), and the best quarter is
Q1 (634 units; the quarter averages rank the same way).

In [ ]:
# Best Month - highest total sales
best_month = df_final.loc[df_final["Total_Sales"].idxmax()]

# Best Product - largest annual total
product_totals = df_final[pcols].sum()
best_product = product_totals.idxmax()

# Best Quarter - largest quarterly total volume
quarter_totals = df_final.groupby("Quarter")["Total_Sales"].sum()
best_quarter = quarter_totals.idxmax()

print("Best Month   :", best_month["Month"], f"({int(best_month['Total_Sales'])} units)")
print("  breakdown :", {c: int(best_month[c]) for c in pcols})
print("Best Product :", best_product, f"({int(product_totals.max())} units over the year)")
print("Best Quarter :", best_quarter, f"({int(quarter_totals.max())} units)")

print("\nAnnual totals per product:")
print(product_totals.sort_values(ascending=False).to_string())
print("\nQuarterly totals:", quarter_totals.astype(int).to_dict())
print("Quarterly means :", df_final.groupby("Quarter")["Total_Sales"].mean().round(2).to_dict())

### 5. Annotated Stacked Bar - Monthly Volume with the Best Month Marked

The handout asks for the stacked bar of monthly volume with the best month annotated. The
annotation is drawn above the tallest stack (Feb, 226 units,
driver: Product_B at 69 units), and the average line gives the
annotation its scale of reference.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
bottom = np.zeros(len(df_final))
for col in pcols:
    ax.bar(df_final["Month"], df_final[col], bottom=bottom, label=col)
    bottom += df_final[col].to_numpy()

ax.axhline(df_final["Total_Sales"].mean(), color="black", linestyle="--", linewidth=1.2,
           label=f"Mean {df_final['Total_Sales'].mean():.1f}")
ax.annotate(f"Best month: {best_month['Month']}\n{int(best_month['Total_Sales'])} units",
            xy=(int(df_final["Total_Sales"].idxmax()), best_month["Total_Sales"]),
            xytext=(int(df_final["Total_Sales"].idxmax()) - 2.4, best_month["Total_Sales"] + 18),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontweight="bold")
ax.set_title("Monthly Total Sales Volume (stacked) - best month annotated", fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Units sold")
ax.set_ylim(0, df_final["Total_Sales"].max() * 1.16)
ax.legend(ncol=5, fontsize=9)
plt.tight_layout()
plt.show()

### 6. Column-Name Aliases (Handout vs Notebook vs Vault)

The same project is described with three different column vocabularies. The handout names the
growth column `Month_over_Month_Growth` and the best/worst product columns `Max_Sales_Product` /
`Min_Sales_Product`; this notebook uses `MoM_Growth`, `Max_Product`, `Min_Product`; the vault
tutorial `03.5` uses `MoM_Growth_Pct`, `Max_Sales_Product`, `Min_Sales_Product`. None of the three
is wrong - they are aliases for the same computed columns - but any shared code or exam answer
must say which vocabulary it uses. The mapping is recorded here once, as a table:

| Concept | Handout (PDF) | This notebook | Vault `03.5` |
| :--- | :--- | :--- | :--- |
| Month-over-month growth | `Month_over_Month_Growth` | `MoM_Growth` | `MoM_Growth_Pct` |
| Product with the highest monthly sales | `Max_Sales_Product` | `Max_Product` | `Max_Sales_Product` |
| Product with the lowest monthly sales | `Min_Sales_Product` | `Min_Product` | `Min_Sales_Product` |

### 7. Conclusions (the Three Handout Questions)

**1. Which product contributes the most to overall sales, and how stable is it?**
Product_A ends the year at 870 units (38% of the 2287-unit year), ahead of
Product_C (474) and Product_B (634); Product_D trails at 309. Its month-to-month
range is wide - single months from 57 to 92 units - so the lead is a total over the
year, not a steady rule: Product_A has the largest quarterly total in Q1, Q3 and Q4,
but Q2 belongs to Product_C.

**2. Is there a seasonal pattern or trend in total sales?**
The 2287-unit year splits as Q1 634, Q2 540, Q3 568, Q4 545: a strong
first half (1174 units, 51% of the year), a soft month 11 (Nov,
-27.4% MoM) and Oct's rebound (29.3% MoM). The Q1 peak (634 units) fades through
Q2 (540) to a flat second half, with Q3 recovering to 568 before the year closes soft. With 12
seeded observations this is a sample pattern, not a forecast.

**3. What drives the best month?**
Feb's 226 units come from Product_B
(69 units, its best month of the year) with every product above its own
average, so the peak is broad-based rather than one product's spike. On the same logic the weakest
month (Nov, 154 units) has three
products below their own averages - the two extremes are the same story read from both ends.